# Flight Price Regression
XGBoost + id-neighbour features, including the full random-search for hyperparameters.

## Imports

In [1]:
import numpy as np
import pandas as pd
import warnings
import os

from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# needs xgboost installed (pip install xgboost)
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

## Config

In [2]:
SEED = 42
# csv files are in the same folder as the notebook, so '.'
DATA_DIR = "."

# folds for cross validation
N_SPLITS = 5
# number of random-search trials, more trials = better chance of a good config but slower
N_ITER_XGB = 30

# id-neighbour trick, set False to see the model without it
USE_ID_NEIGHBOURS = True
# neighbours taken on each side
K_NEIGHBOURS = 1

# metric used to pick the best trial (assuming the leaderboard is R2)
TARGET_METRIC = "r2"
# log target made RMSE/R2 worse in our tests, so it stays off
TRY_LOG_TARGET = False

## Load data

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

# price as float for the metrics
y = train["price"].values.astype(float)

# ids only used for neighbour lookup and the submission, not as a feature
ids_tr = train["id"].values
ids_te = test["id"].values

print(train.shape, test.shape)

(18076, 12) (4520, 11)


## Feature engineering

In [4]:
# stops are text, map them to 0/1/2
STOPS_MAP = {
    "zero": 0,
    "one": 1,
    "two_or_more": 2
}

def make_features(df):

    d = df.copy()

    # flight is like 'UK 811', split into carrier and number
    d["carrier"] = d["flight"].str.split().str[0]
    d["flight_num"] = d["flight"].str.split().str[1].astype(int)

    # source and destination as one route string
    d["route"] = (
        d["source_city"] + "_" +
        d["destination_city"]
    )

    d["stops_n"] = d["stops"].map(STOPS_MAP)

    d["is_business"] = (
        d["flight_class"] == "Business"
    ).astype(int)

    # rough length of each leg
    d["dur_per_stop"] = (
        d["duration"] /
        (d["stops_n"] + 1)
    )

    # prices jump close to departure, flag the last 7 days
    d["last_week"] = (
        d["days_left"] <= 7
    ).astype(int)

    # business fares react to days_left differently from economy
    d["class_x_dl"] = (
        d["is_business"] *
        d["days_left"]
    )

    # drop id from the features, it is only an identifier
    return d.drop(columns=["id"])

# rows with same class, days_left and route usually have similar prices
def group_key(df):

    return (
        df["flight_class"]
        + "|"
        + df["days_left"].astype(str)
        + "|"
        + df["source_city"]
        + "_"
        + df["destination_city"]
    ).values

X_all = make_features(train).drop(columns=["price"])
X_te = make_features(test)

key_tr = group_key(train)
key_te = group_key(test)

## Matrix for XGBoost

In [5]:
CAT_COLS = [
    "airline",
    "flight",
    "source_city",
    "departure_time",
    "stops",
    "arrival_time",
    "destination_city",
    "flight_class",
    "carrier",
    "route"
]

# 'flight' has too many unique values to one-hot, so only the rest get encoded
LOW_CARD = [c for c in CAT_COLS if c != "flight"]

def xgb_matrix(X):

    return pd.get_dummies(
        X.drop(columns=["flight"]),
        columns=LOW_CARD,
        dtype=int
    )

Xx_all = xgb_matrix(X_all)

# align test columns with train (anything missing becomes 0)
Xx_te = xgb_matrix(X_te).reindex(
    columns=Xx_all.columns,
    fill_value=0
)

## Id neighbour features
The files are stored in sorted order and train/test rows are mixed, so the nearest train rows by id usually share class, days_left and route and have almost the same price.

In [6]:
def nb_features(
    ids_ref,
    y_ref,
    key_ref,
    ids_q,
    key_q,
    self_in,
    k=1
):

    # sort reference rows by id so prev/next means something
    o = np.argsort(ids_ref)

    sid = ids_ref[o]
    sy = y_ref[o]
    sk = np.asarray(key_ref)[o]

    # where each query id falls in the sorted list
    pos = np.searchsorted(
        sid,
        ids_q
    )

    out = {}

    for j in range(1, k + 1):

        for side, sign in (
            ("prev", -1),
            ("next", 1)
        ):

            # self_in=True: queries are the reference rows themselves, so skip the row itself
            idx = (
                pos + sign * j
                if self_in
                else (
                    (pos - j)
                    if sign < 0
                    else (pos + j - 1)
                )
            )

            # don't run off the ends of the list
            ok = (
                (idx >= 0)
                &
                (idx < len(sid))
            )

            ic = np.clip(
                idx,
                0,
                len(sid) - 1
            )

            # neighbour price, id distance, and whether it shares class/days_left/route
            out[f"nb_{side}{j}"] = np.where(
                ok,
                sy[ic],
                np.nan
            )

            out[f"nb_d{side}{j}"] = np.where(
                ok,
                np.abs(sid[ic] - ids_q),
                np.nan
            )

            out[f"nb_same{side}{j}"] = np.where(
                ok,
                (
                    sk[ic] ==
                    np.asarray(key_q)
                ).astype(float),
                np.nan
            )

    df = pd.DataFrame(out)

    # mean of the two neighbour prices
    df["nb_mean"] = (
        df[["nb_prev1", "nb_next1"]]
        .mean(axis=1)
    )

    return df

In [7]:
def add_nb(X, nb):

    if USE_ID_NEIGHBOURS:

        return pd.concat(
            [
                X.reset_index(drop=True),
                nb.reset_index(drop=True)
            ],
            axis=1
        )

    return X.reset_index(drop=True)

## Cross-validation folds

In [8]:
# neighbour features are built inside each fold from the training part only,
# so validation prices never leak into the features
def fold_data(tr_i, va_i):

    if USE_ID_NEIGHBOURS:

        n_tr = nb_features(
            ids_tr[tr_i],
            y[tr_i],
            key_tr[tr_i],
            ids_tr[tr_i],
            key_tr[tr_i],
            True,
            K_NEIGHBOURS
        )

        n_va = nb_features(
            ids_tr[tr_i],
            y[tr_i],
            key_tr[tr_i],
            ids_tr[va_i],
            key_tr[va_i],
            False,
            K_NEIGHBOURS
        )

    else:

        n_tr = None
        n_va = None

    return (
        add_nb(Xx_all.iloc[tr_i], n_tr),
        add_nb(Xx_all.iloc[va_i], n_va)
    )

# same folds are reused for every trial so trials are comparable
kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

FOLDS = list(kf.split(Xx_all))

# computed once up front, saves a lot of time during the search
FOLD_DATA = [
    fold_data(a, b)
    for a, b in FOLDS
]

## Metrics

In [9]:
def score(y_true, y_pred):

    return {
        "rmse": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),
        "mae": mean_absolute_error(
            y_true,
            y_pred
        ),
        "r2": r2_score(
            y_true,
            y_pred
        )
    }

# lower is better, so R2 gets a minus sign
def sort_key(s):

    return (
        -s["r2"]
        if TARGET_METRIC == "r2"
        else s[TARGET_METRIC]
    )

# log-target helpers (not used while TRY_LOG_TARGET is False)
def fwd(v, log):

    return (
        np.log1p(v)
        if log
        else v
    )

def inv(v, log):

    return (
        np.clip(
            np.expm1(v),
            0,
            None
        )
        if log
        else np.clip(
            v,
            0,
            None
        )
    )

# one random generator so the search is reproducible
rng = np.random.RandomState(SEED)

## Random search for hyperparameters

In [10]:
# draws one random set of hyperparameters
def sample_xgb():

    return dict(
        learning_rate=float(
            rng.choice(
                [0.02, 0.03, 0.05, 0.08]
            )
        ),

        max_depth=int(
            rng.randint(3, 11)
        ),

        min_child_weight=float(
            rng.choice(
                [1, 2, 3, 5, 8]
            )
        ),

        subsample=float(
            rng.uniform(0.6, 1.0)
        ),

        colsample_bytree=float(
            rng.uniform(0.5, 1.0)
        ),

        reg_lambda=float(
            10 ** rng.uniform(-1, 1.5)
        ),

        reg_alpha=float(
            10 ** rng.uniform(-3, 1)
        ),

        gamma=float(
            rng.choice(
                [0, 0.1, 0.5, 1]
            )
        ),

        log_target=False
    )

In [11]:
# 5-fold CV for one hyperparameter set, with early stopping on the validation fold
def cv_xgb(params):

    p = dict(params)

    log = p.pop("log_target")

    oof = np.zeros(len(y))

    iters = []

    for (tr_i, va_i), (Xt, Xv) in zip(
        FOLDS,
        FOLD_DATA
    ):

        # n_estimators is just a high cap, early stopping decides the real number
        model = XGBRegressor(
            n_estimators=4000,
            early_stopping_rounds=100,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            **p
        )

        model.fit(
            Xt,
            fwd(y[tr_i], log),
            eval_set=[
                (
                    Xv,
                    fwd(y[va_i], log)
                )
            ],
            verbose=False
        )

        oof[va_i] = inv(
            model.predict(Xv),
            log
        )

        # remember how many trees each fold needed
        iters.append(
            model.best_iteration + 1
        )

    s = score(y, oof)

    # average best iteration + 10% (final model trains on more data)
    return (
        s,
        int(np.mean(iters) * 1.1)
    )

## Run the search
Prints every trial (RMSE, MAE, R2) and keeps the best one.

In [12]:
best_xgb = None
xgb_log = []

# this is the actual random search, one CV run per trial
for i in range(N_ITER_XGB):

    params = sample_xgb()

    s, n_est = cv_xgb(params)

    # keep every trial for the report
    xgb_log.append({
        **params,
        "n_estimators": n_est,
        **s
    })

    # keep the best trial so far
    if (
        best_xgb is None
        or
        sort_key(s)
        <
        sort_key(best_xgb["score"])
    ):

        best_xgb = {
            "params": params,
            "n_estimators": n_est,
            "score": s
        }

    print(
        f"Trial {i+1}/{N_ITER_XGB} | "
        f"RMSE={s['rmse']:.2f} | "
        f"MAE={s['mae']:.2f} | "
        f"R2={s['r2']:.5f}"
    )

# save all trials, handy evidence for the tuning part of the report
pd.DataFrame(
    xgb_log
).to_csv(
    "xgb_tuning_log.csv",
    index=False
)

print("\nBEST XGB")
print(best_xgb)

Trial 1/30 | RMSE=98.76 | MAE=26.90 | R2=0.80204
Trial 2/30 | RMSE=100.66 | MAE=28.05 | R2=0.79435
Trial 3/30 | RMSE=98.91 | MAE=24.89 | R2=0.80141
Trial 4/30 | RMSE=100.50 | MAE=31.07 | R2=0.79498
Trial 5/30 | RMSE=99.44 | MAE=24.83 | R2=0.79927
Trial 6/30 | RMSE=103.24 | MAE=32.70 | R2=0.78366
Trial 7/30 | RMSE=99.35 | MAE=26.81 | R2=0.79965
Trial 8/30 | RMSE=98.13 | MAE=26.67 | R2=0.80455
Trial 9/30 | RMSE=97.24 | MAE=23.65 | R2=0.80806
Trial 10/30 | RMSE=98.09 | MAE=24.36 | R2=0.80469
Trial 11/30 | RMSE=97.68 | MAE=28.54 | R2=0.80633
Trial 12/30 | RMSE=99.22 | MAE=23.21 | R2=0.80019
Trial 13/30 | RMSE=98.29 | MAE=27.51 | R2=0.80390
Trial 14/30 | RMSE=99.26 | MAE=24.23 | R2=0.80003
Trial 15/30 | RMSE=96.42 | MAE=23.62 | R2=0.81128
Trial 16/30 | RMSE=99.61 | MAE=24.36 | R2=0.79861
Trial 17/30 | RMSE=97.53 | MAE=23.78 | R2=0.80693
Trial 18/30 | RMSE=99.93 | MAE=28.59 | R2=0.79729
Trial 19/30 | RMSE=97.95 | MAE=24.70 | R2=0.80524
Trial 20/30 | RMSE=96.73 | MAE=23.25 | R2=0.81008
Trial 

## Final features (all train rows)

In [13]:
# for the final model, neighbours come from ALL train rows
nb_tr = nb_features(
    ids_tr,
    y,
    key_tr,
    ids_tr,
    key_tr,
    True,
    K_NEIGHBOURS
)

# test rows look up their neighbours among train rows
nb_te = nb_features(
    ids_tr,
    y,
    key_tr,
    ids_te,
    key_te,
    False,
    K_NEIGHBOURS
)

FX_tr = add_nb(Xx_all, nb_tr)
FX_te = add_nb(Xx_te, nb_te)

## Final training

In [14]:
preds = []

# train on all data with 3 seeds and average, slightly more stable
for seed in [42, 7, 2024]:

    p = dict(best_xgb["params"])

    # log_target is not an XGBoost argument, remove it before passing the rest
    p.pop("log_target")

    model = XGBRegressor(
        n_estimators=best_xgb["n_estimators"],
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
        **p
    )

    model.fit(
        FX_tr,
        y
    )

    preds.append(
        model.predict(FX_te)
    )

# average of the 3 models
final_pred = np.mean(
    preds,
    axis=0
)

## Submission

In [15]:
# round to whole numbers and make sure nothing is negative
sub = pd.DataFrame({
    "id": ids_te,
    "price": np.clip(
        np.round(final_pred),
        0,
        None
    ).astype(int)
})

sub.to_csv(
    "submission.csv",
    index=False
)

print("\nSaved submission.csv")
print(sub.head())


Saved submission.csv
      id  price
0  18395     63
1  11771    156
2  22326    364
3  18332    138
4  11021    585
